# Post-quantum LMS/HSS user authentication

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

Hardened SHAKE plus software verification checks hash-based signatures. This addresses quantum-computer attacks on signature algorithms, not electromagnetic or quantum interference. Runtime vectors use H5/W2; their HSS wrapper has one level. Boot signing is a separate two-level H15 demonstration.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
from hss_ref import load_vectors, verify_lms, verify_hss
vectors = load_vectors(ROOT / "host/inputs/hss.c")
for name, verifier in [("Lms", verify_lms), ("Hss", verify_hss)]:
    v = vectors[name]; message, signature, public_key = v["message"], v["signature"], v["public_key"]
    assert verifier(message, signature, public_key)
    changed = signature[:32] + bytes([signature[32] ^ 1]) + signature[33:]
    assert not verifier(message, changed, public_key)
    start = time.perf_counter_ns()
    for _ in range(20): assert verifier(message, signature, public_key)
    print(name, "Python mean us:", (time.perf_counter_ns()-start)/20/1000)
display(rows[rows.algorithm.isin(["LMS-SHAKE256", "HSS-SHAKE256"])])

Lms Python mean us: 598.21
Hss Python mean us: 518.24


,algorithm,case,size_bytes,repeats,copy_cycles,setup_cycles,api_cycles,check_cycles,scu35_api_mean_us,scu35_full_mean_us,scu35_operation_mean_us,scu35_min_us,scu35_max_us,status
20,LMS-SHAKE256,valid,56,10,0,4570,12597947,0,12597.947,12602.517,12597.947,12597.79,12598.10,0
21,LMS-SHAKE256,bad_signature,56,10,0,4570,11763269,0,11763.269,11767.839,11763.269,11763.19,11763.36,171
22,LMS-SHAKE256,bad_message,56,10,0,4570,12418531,0,12418.531,12423.101,12418.531,12418.41,12418.59,171
23,LMS-SHAKE256,bad_key,56,10,0,4570,12597853,0,12597.853,12602.423,12597.853,12597.72,12598.01,171
24,HSS-SHAKE256,valid,56,10,0,4570,11839031,0,11839.031,11843.601,11839.031,11838.89,11839.15,0
25,HSS-SHAKE256,bad_signature,56,10,0,4570,12642577,0,12642.577,12647.147,12642.577,12642.46,12642.68,171
26,HSS-SHAKE256,bad_message,56,10,0,4570,12314996,0,12314.996,12319.566,12314.996,12314.84,12315.14,171
27,HSS-SHAKE256,bad_key,56,10,0,4570,11839023,0,11839.023,11843.593,11839.023,11838.96,11839.06,171


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).